## Imports and Directory settings


In [2]:
# Turn on autoreload
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [16]:
import sys
import warnings
from pathlib import Path

import numpy as np
from loguru import logger

from mnutils import GESeries
from mnutils.fitting.AMARES import plot_amares_fitting
import xmris
import pandas as pd
import xarray as xr

In [4]:
# Show all RuntimeWarnings as errors to catch them during testing
warnings.simplefilter("error", RuntimeWarning)

## Set logger to debug


In [5]:
logger.remove()
logger.add(sys.stdout, level="INFO")

1

# AMARES fitting on unlocalised and localised MRSI data

This notebook fits the same 2H prior-knowledge model (`tests/datasets/2H_4peaks.csv`, DHO/Glucose/Glx/Lac) to two series from the `HeVo-18` dataset:

- an **unlocalised** single-voxel MRS series (`MRSSeries`, fit via `fit_average_fid`)
- a **localised** spatial MRSI series (`MRSISeries`, fit via `fit_all_voxels`)

and asserts that both fits converge well (finite, reasonably small chi-square; few/no bad fits).


In [6]:
DATA_FOLDER = Path.cwd() / "datasets" / "HeVo-18" / "data"
pk_file = str(Path.cwd() / "datasets" / "2H_4peaks.csv")
pk = pd.read_csv(pk_file, index_col=0)

mrs = GESeries.MRSSeries(DATA_FOLDER, 6)  # unlocalised
mrsi = GESeries.MRSISeries(DATA_FOLDER, 8)  # localised

init_params = {"priorknowledgefile": pk_file}

# Display function

In [18]:
from itertools import product
import cmap
import matplotlib.pyplot as plt


def plot_fit_spectrum(
    fit: xr.Dataset,
    figsize=None,
    xlim=(10, -2),
    colormap_name="tab10",
    raw_alpha=0.4,
    residual_color="lime",
    show_legend=True,
    max_subplots=10,
    subplot_cols=3,
):
    """Plot phased raw, fitted metabolite spectra, and residuals.

    Supports a single fit or a small grid of fits from a 2D dataset.
    """
    required_vars = {"raw_data", "fit_data", "residuals", "chem_shift"}
    missing_vars = required_vars.difference(set(fit.data_vars).union(set(fit.coords)))
    if missing_vars:
        raise ValueError(
            f"fit is missing required variables/coords: {sorted(missing_vars)}"
        )

    cm = cmap.Colormap(colormap_name)

    plot_dims = [dim for dim in fit.raw_data.dims if dim != "time"]
    if plot_dims:
        selectors = [
            dict(zip(plot_dims, idxs))
            for idxs in product(*[range(fit.sizes[dim]) for dim in plot_dims])
        ]
    else:
        selectors = [{}]

    total_panels = len(selectors)
    if total_panels > max_subplots:
        selectors = selectors[:max_subplots]

    n_panels = len(selectors)

    def _format_title(selector):
        if not selector:
            return ""
        parts = []
        for dim, idx in selector.items():
            value = fit.coords[dim].isel({dim: idx}).item()
            if isinstance(value, (float, np.floating)):
                value_str = f"{float(value):g}"
            else:
                value_str = str(value)
            if dim == "timepoint":
                value_str += " min"
            parts.append(f"{dim}={value_str}")
        return "\n".join(parts)

    def _plot_one(ax, fit_sel, selector):
        spec_raw = fit_sel.raw_data.xmr.to_spectrum().xmr.autophase().xmr.to_ppm()
        phase_kwargs = {
            "p0": spec_raw.attrs["phase_p0"],
            "p1": spec_raw.attrs["phase_p1"],
            "pivot": spec_raw.attrs["phase_pivot"],
        }
        spec_fit = (
            fit_sel.fit_data.xmr.to_spectrum().xmr.phase(**phase_kwargs).xmr.to_ppm()
        )
        spec_res = (
            fit_sel.residuals.xmr.to_spectrum().xmr.phase(**phase_kwargs).xmr.to_ppm()
        )

        ax.plot(
            spec_raw.coords["chemical_shift"],
            spec_raw.real,
            color="black",
            alpha=raw_alpha,
            label="Raw Data",
        )

        ax.plot(
            spec_fit.coords["chemical_shift"],
            spec_fit.sum("Metabolite").real,
            color="lightcoral",
            linewidth=2,
            label="Total Fit",
        )

        for idx, (metabolite, met_spec) in enumerate(spec_fit.groupby("Metabolite")):
            color = cm(idx % cm.num_colors)
            ax.plot(
                met_spec.coords["chemical_shift"],
                met_spec.squeeze().real,
                color=color,
                linewidth=1.5,
                label=f"{metabolite} Fit",
            )

            chem_shift_value = fit_sel.chem_shift.sel(
                Metabolite=metabolite
            ).values.squeeze()
            ax.axvline(x=chem_shift_value, color=color, linestyle="dashed", alpha=0.3)
            ax.text(
                chem_shift_value - 0.1,
                ax.get_ylim()[1] * 0.9,
                f"{metabolite}\n({chem_shift_value:.2f})",
                color=color,
                fontsize=9,
                ha="left",
            )

        ax.plot(
            spec_res.coords["chemical_shift"],
            spec_res.real,
            color=residual_color,
            linestyle="dotted",
            label="Residual",
        )

        ax.set_xlim(*xlim)
        ax.set_xlabel("Chemical Shift (ppm)")
        ax.set_ylabel("Intensity (a.u.)")

        title = _format_title(selector)
        if title:
            ax.set_title(title, fontsize=10)

        if show_legend:
            ax.legend(fontsize=8)

    if figsize is None:
        if n_panels == 1:
            figsize = (8, 5)
        else:
            ncols = min(subplot_cols, n_panels)
            nrows = int(np.ceil(n_panels / ncols))
            figsize = (5.5 * ncols, 4.5 * nrows)

    if n_panels == 1:
        fig, ax = plt.subplots(figsize=figsize)
        _plot_one(ax, fit.isel(**selectors[0]).squeeze(drop=True), selectors[0])  # type: ignore
        fig.tight_layout()
        return fig, ax

    ncols = min(subplot_cols, n_panels)
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False, sharex=True)
    axes_flat = axes.ravel()

    for ax, selector in zip(axes_flat, selectors):
        _plot_one(ax, fit.isel(**selector).squeeze(drop=True), selector)  # type: ignore

    for ax in axes_flat[n_panels:]:
        ax.set_visible(False)

    if total_panels > max_subplots:
        fig.suptitle(
            f"Showing first {max_subplots} of {total_panels} fits", fontsize=12
        )

    return fig, axes_flat[:n_panels]

## Unlocalised fit


In [12]:
mrs_result = mrs.fit_average_fid(pk)

# chisqr = mrs_result.out_obj.chisqr
# redchi = mrs_result.out_obj.redchi
# logger.info(f"Unlocalised fit: chisqr={chisqr:.4g}, redchi={redchi:.4g}")

# assert np.isfinite(chisqr)
# assert np.isfinite(redchi)
# assert redchi < 10, f"Unlocalised fit did not converge well: redchi={redchi:.4g}"

# fitted_amplitudes = mrs_result.amares_to_plot_pd["Amplitude"]
# assert np.all(np.isfinite(fitted_amplitudes)), "Non-finite fitted amplitude(s)"
# assert np.all(fitted_amplitudes >= 0), "Negative fitted amplitude(s)"

In [ ]:
xmris.fit_amares

In [21]:
import pyAMARES

In [ ]:
pyAMARES.uninterleave
pyAMARES.multieq6

In [20]:
mrs_result

<xarray.Dataset> Size: 99kB
Dimensions:     (time: 2048, metabolite: 4, parameter: 4)
Coordinates:
  * time        (time) float64 16kB 0.0 0.0002 0.0004 ... 0.409 0.4092 0.4094
  * metabolite  (metabolite) object 32B 'DHO' 'Glucose' 'Glx' 'Lac'
  * parameter   (parameter) <U10 160B 'amplitude' 'chem_shift' ... 'phase'
Data variables:
    data        (time) complex64 16kB (16723401-8241802.5j) ... (-461.39062-3...
    fit         (time) complex128 33kB (27852096.713465285-13429472.988861509...
    residuals   (time) complex128 33kB (-11128695.713465285+5187670.488861509...
    amplitude   (metabolite) float64 32B 2.501e+07 2.444e+06 1.216e+06 2.252e+06
    chem_shift  (metabolite) float64 32B 4.659 3.733 2.1 0.8228
    linewidth   (metabolite) float64 32B 15.15 15.15 15.15 15.15
    phase       (metabolite) float64 32B -25.74 -25.74 -25.74 -25.74
    snr         (metabolite) float64 32B 50.92 4.976 2.476 4.586
    crlb        (metabolite, parameter) float64 128B 0.6224 10.8 ... 0.7359 1.02
    sd          (metabolite, parameter) float64 128B 1.472e+05 ... 0.2483
    fit_status  int8 1B 0
Attributes:
    field_strength:          3.0
    reference_frequency:     19.612622
    carrier_ppm:             4.68
    units:                   a.u.
    spectral_width:          5000.0
    deadtime:                0.000706
    amares_amplitude_scale:  38466928.0

In [19]:
plot_fit_spectrum(mrs_result)

ValueError: fit is missing required variables/coords: ['fit_data', 'raw_data']

## Localised fit

`testing_mode=True` limits fitting to the first 100 SNR-passing voxels so the notebook runs quickly.


In [ ]:
badfit_ids = mrsi.fit_all_voxels(init_params=init_params, testing_mode=True)

SNR_mask = mrsi.get_SNR_mask()
num_fitted_voxels = min(int(SNR_mask.sum()), 100)
logger.info(
    f"Localised fit: {badfit_ids.size}/{num_fitted_voxels} voxels flagged as bad fits"
)

assert num_fitted_voxels > 0, "No SNR-passing voxels were fitted"
assert badfit_ids.size < 0.2 * num_fitted_voxels, (
    f"Too many bad fits: {badfit_ids.size}/{num_fitted_voxels}"
)

chisqr_map = mrsi.goodness_of_fit_maps["chisqr"]
fitted_chisqr = chisqr_map[~np.isnan(chisqr_map)]
assert fitted_chisqr.size == num_fitted_voxels
assert np.all(np.isfinite(fitted_chisqr)), "Non-finite chisqr in fitted voxels"

In [ ]:
mrsi.visualize_goodness_of_fit_map("chisqr")

In [ ]:
mrsi.visualize_fitted_metabolite_map("DHO")

Both the unlocalised and localised AMARES fits converged well.
